# Pakistan Stock Exchange (KSE-100) Analysis 2020-2025
### 42,000+ Daily Records | 28 Companies | EDA + Machine Learning
**Author:** Hassan Ali | [Kaggle: hassanali789](https://www.kaggle.com/hassanali789)

This notebook provides a complete analysis of Pakistan Stock Exchange data covering 28 top KSE-100 companies across 8 sectors from 2020 to 2025. We explore price trends, sector performance, volatility, correlations, and build a stock return prediction model.

**Source:** Yahoo Finance

---


## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("All libraries loaded successfully!")

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/psx-kse100-stocks-2020-2025/psx_stocks_2020_2025.csv",
                 parse_dates=["date"])

df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month

print(f"Shape          : {df.shape}")
print(f"Date range     : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Companies      : {df['ticker'].nunique()}")
print(f"Sectors        : {df['sector'].nunique()}")
print()
print(df[['ticker','company','sector']].drop_duplicates().sort_values('sector').to_string(index=False))
df.head(10)

## 3. Statistical Summary

In [ ]:
print("=== Overall Statistics ===")
print(df[["open","high","low","close","volume","daily_return_pct"]].describe().round(2))

print("\n=== Companies per Sector ===")
print(df.groupby("sector")["ticker"].nunique().sort_values(ascending=False))

## 4. Closing Price Trends by Sector

In [ ]:
sectors = sorted(df["sector"].unique())
fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()

for i, sector in enumerate(sectors):
    sector_df = df[df["sector"] == sector]
    for ticker in sector_df["ticker"].unique():
        t_df = sector_df[sector_df["ticker"] == ticker].sort_values("date")
        axes[i].plot(t_df["date"], t_df["close"], linewidth=1.2,
                     label=ticker.replace(".KA",""))
    axes[i].set_title(f"{sector} Sector", fontsize=12, fontweight="bold")
    axes[i].set_ylabel("Close Price (PKR)")
    axes[i].legend(fontsize=7, loc="upper left")
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.suptitle("KSE-100 Stock Prices by Sector (2020-2025)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("price_trends_by_sector.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Best and Worst Performing Stocks (2020-2025)

In [ ]:
returns = []
for ticker in df["ticker"].unique():
    t_df = df[df["ticker"] == ticker].sort_values("date")
    if len(t_df) < 10:
        continue
    start_price = t_df.iloc[0]["close"]
    end_price   = t_df.iloc[-1]["close"]
    total_return = ((end_price - start_price) / start_price) * 100
    returns.append({
        "ticker"     : ticker.replace(".KA",""),
        "company"    : t_df.iloc[0]["company"],
        "sector"     : t_df.iloc[0]["sector"],
        "return_pct" : round(total_return, 2)
    })

returns_df = pd.DataFrame(returns).sort_values("return_pct", ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ["#43A047" if r > 0 else "#E53935" for r in returns_df["return_pct"]]
bars = ax.barh(returns_df["ticker"], returns_df["return_pct"],
               color=colors, edgecolor="white", linewidth=0.3)
ax.bar_label(bars, fmt="%.0f%%", padding=3, fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Total Return by Stock (2020-2025)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Total Return (%)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("total_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print(returns_df.to_string(index=False))

## 6. Sector-wise Average Performance

In [ ]:
sector_ret = returns_df.groupby("sector")["return_pct"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#43A047" if r > 0 else "#E53935" for r in sector_ret.values]
bars = ax.bar(sector_ret.index, sector_ret.values,
              color=colors, edgecolor="white", linewidth=0.3)
ax.bar_label(bars, fmt="%.0f%%", padding=3, fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Average Total Return by Sector (2020-2025)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Average Return (%)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("sector_performance.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Volatility Analysis

In [ ]:
volatility = df.groupby("ticker")["daily_return_pct"].std().sort_values(ascending=False).reset_index()
volatility["ticker"] = volatility["ticker"].str.replace(".KA","")
volatility.columns = ["ticker","volatility"]

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(volatility)))
bars = ax.bar(volatility["ticker"], volatility["volatility"],
              color=colors, edgecolor="white", linewidth=0.3)
ax.set_title("Daily Return Volatility by Stock (Std Dev %)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Volatility (Std Dev of Daily Return %)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("volatility.png", dpi=150, bbox_inches="tight")
plt.show()

print("Most volatile :", volatility.iloc[0]["ticker"], f"({volatility.iloc[0]['volatility']:.2f}%)")
print("Most stable   :", volatility.iloc[-1]["ticker"], f"({volatility.iloc[-1]['volatility']:.2f}%)")

## 8. Stock Correlation Heatmap

In [ ]:
pivot = df.pivot_table(values="close", index="date", columns="ticker")
pivot.columns = pivot.columns.str.replace(".KA","")
returns_pivot = pivot.pct_change().dropna()
corr = returns_pivot.corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=-1, vmax=1, linewidths=0.3, linecolor="white",
            annot_kws={"size": 7}, ax=ax)
ax.set_title("Stock Return Correlation Matrix", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Trading Volume Analysis

In [ ]:
monthly_vol = df.groupby(["year","month"])["volume"].sum().reset_index()
monthly_vol["period"] = pd.to_datetime(monthly_vol[["year","month"]].assign(day=1))

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(monthly_vol["period"], monthly_vol["volume"]/1e6,
       color="#1565C0", edgecolor="white", linewidth=0.2, width=20)
ax.set_title("Total Monthly Trading Volume — All Stocks (2020-2025)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Volume (Millions)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.savefig("volume_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. COVID-19 Impact on KSE-100 (2020)

In [ ]:
covid_df = df[df["year"] == 2020].copy()

fig, ax = plt.subplots(figsize=(13, 6))
for ticker in covid_df["ticker"].unique():
    t_df = covid_df[covid_df["ticker"] == ticker].sort_values("date")
    if t_df.empty or t_df.iloc[0]["close"] == 0:
        continue
    normalized = (t_df["close"] / t_df.iloc[0]["close"]) * 100
    ax.plot(t_df["date"], normalized, linewidth=1, alpha=0.4,
            label=ticker.replace(".KA",""))

ax.axhline(100, color="black", linewidth=1, linestyle="--", label="Base (Jan 2020)")
ax.axvline(pd.Timestamp("2020-03-23"), color="red", linewidth=1.5,
           linestyle="--", label="PSX Circuit Breaker (Mar 23)")
ax.set_title("Stock Performance During COVID-19 Crash (2020)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Normalized Price (Base=100)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig("covid_impact.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Monthly Returns Heatmap

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

monthly_ret = df.groupby(["year","month"])["daily_return_pct"].mean().reset_index()
pivot_ret = monthly_ret.pivot(index="year", columns="month", values="daily_return_pct")
pivot_ret.columns = month_names[:len(pivot_ret.columns)]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot_ret, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Avg Daily Return (%)"}, ax=ax)
ax.set_title("Average Daily Return by Month and Year (%)", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("monthly_returns_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Feature Engineering for Machine Learning

In [ ]:
df_ml = df.copy().sort_values(["ticker","date"])

df_ml["ma_7"]          = df_ml.groupby("ticker")["close"].transform(lambda x: x.rolling(7).mean())
df_ml["ma_30"]         = df_ml.groupby("ticker")["close"].transform(lambda x: x.rolling(30).mean())
df_ml["ma_7_30_ratio"] = df_ml["ma_7"] / df_ml["ma_30"]
df_ml["volatility_7"]  = df_ml.groupby("ticker")["daily_return_pct"].transform(lambda x: x.rolling(7).std())
df_ml["return_lag1"]   = df_ml.groupby("ticker")["daily_return_pct"].shift(1)
df_ml["return_lag3"]   = df_ml.groupby("ticker")["daily_return_pct"].shift(3)
df_ml["return_lag5"]   = df_ml.groupby("ticker")["daily_return_pct"].shift(5)
df_ml["volume_change"] = df_ml.groupby("ticker")["volume"].pct_change()
df_ml["price_range"]   = (df_ml["high"] - df_ml["low"]) / df_ml["close"] * 100
df_ml["next_return"]   = df_ml.groupby("ticker")["daily_return_pct"].shift(-1)
df_ml["target"]        = (df_ml["next_return"] > 0).astype(int)

le = LabelEncoder()
df_ml["sector_enc"] = le.fit_transform(df_ml["sector"])
df_ml = df_ml.dropna()

print(f"ML-ready rows : {len(df_ml):,}")
print(f"Class balance :\n{df_ml['target'].value_counts()}")
df_ml.head()

## 13. Model Training — Predicting Next Day Price Direction

In [ ]:
features = ["close","open","high","low","volume",
            "daily_return_pct","ma_7","ma_30","ma_7_30_ratio",
            "volatility_7","return_lag1","return_lag3","return_lag5",
            "volume_change","price_range","month","sector_enc"]

X = df_ml[features].fillna(0)
y = df_ml["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test size : {X_test.shape[0]:,}")
print()

models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []
trained = {}

for name, model in models.items():
    Xtr = X_train_s if name == "Logistic Regression" else X_train
    Xte = X_test_s  if name == "Logistic Regression" else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    acc = accuracy_score(y_test, preds)
    results.append({"Model": name, "Accuracy": round(acc, 4)})
    trained[name] = (model, preds)
    print(f"{name:25s} -> Accuracy: {acc:.4f}")

## 14. Model Comparison and Feature Importance

In [ ]:
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
best_name  = results_df.iloc[0]["Model"]
best_preds = trained[best_name][1]
best_model = trained[best_name][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(results_df["Model"], results_df["Accuracy"],
                   color=["#43A047","#1E88E5","#FB8C00"],
                   edgecolor="white", linewidth=0.3)
axes[0].bar_label(bars, fmt="%.4f", padding=3, fontsize=10)
axes[0].set_ylim(0.4, 0.7)
axes[0].set_title("Model Accuracy Comparison", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=10)
axes[0].axhline(0.5, color="red", linestyle="--", linewidth=1, label="Random baseline (50%)")
axes[0].legend()

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=features).sort_values()
    axes[1].barh(importances.index, importances.values, color="#66BB6A")
    axes[1].set_title(f"Feature Importance — {best_name}", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best Model: {best_name}")
print(f"Accuracy  : {results_df.iloc[0]['Accuracy']}")

## 15. Confusion Matrix and Classification Report

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Down","Up"],
            yticklabels=["Down","Up"], ax=ax)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("Classification Report:")
print(classification_report(y_test, best_preds, target_names=["Down","Up"]))

## 16. Key Findings and Conclusions

### Market Performance
- Technology sector (Systems Ltd, TRG) delivered the strongest returns over 2020-2025
- Banking sector showed consistent stability with moderate returns
- COVID-19 crash in March 2020 caused significant drawdowns across all stocks
- PSX circuit breaker on March 23, 2020 temporarily halted trading

### Volatility and Risk
- Small-cap and technology stocks showed highest volatility
- Banking stocks (HBL, UBL, MCB) were among the most stable
- High correlation within sectors confirms sector rotation patterns in KSE-100

### Machine Learning
- Predicting stock price direction is challenging — market is largely efficient
- Technical indicators (moving averages, lag returns) are the strongest predictors
- Any accuracy above 50% beats random guessing for directional prediction

---

Dataset by Hassan Ali | hassanali789 on Kaggle
Source: Yahoo Finance
